**This notebook generates the passive iterative reasoning trajectories used for HotpotQA and 2WikiMultiHopQA. Reasoning is generated one step at a time without verifier intervention, followed by repeated final-answer sampling for confidence estimation.**

In [ ]:
!pip install -U transformers trl peft bitsandbytes datasets accelerate rank-bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.7/11.7 MB 147.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 52.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 59.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 67.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 49.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 51.6 MB/s eta 0:00:00
  Attempting uninstall: pyarrow
    Found existing installation: pyarrow 18.1.0
    Uninstalling pyarrow-18.1.0:
      Successfully uninstalled pyarrow-18.1.0
  Attempting uninstall: datasets
    Found existing installation: datasets 4.0.0
    Uninstalling datasets-4.0.0:
      Successfully uninstalled datasets-4.0.0
  Attempting uninstall: transformers
    Found existing installation: transformers 5.13.1
    Uninstalling transformers-5.13.1:
      Successfully uninstalled transformers-5.13.1
  Attempting uninstal

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### **Iterative Generation**


In [ ]:
import os
import re
import json
import random
from collections import Counter

import numpy as np
import torch

from tqdm.auto import tqdm
from datasets import load_dataset
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    StoppingCriteria,
    StoppingCriteriaList,
)


DATA_DIR = "/content/drive/MyDrive/hedge_run"

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
EMBED_ID = "BAAI/bge-small-en-v1.5"

SEED = 42

RUN_DATASETS = [
    "hotpot",
    "2wiki",
]

RETRIEVE_K = 10
MAX_STEPS = 4
STEP_MAX_NEW_TOKENS = 90
FINAL_SAMPLES = 10
FINAL_TEMPERATURE = 0.7
FINAL_TOP_P = 0.95
FINAL_MAX_NEW_TOKENS = 80
EMPTY_FINAL_RETRY_ATTEMPTS = 2
ANSWER_MERGE_SIM = 0.80
SAVE_EVERY = 5
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
print("Loading Qwen...")


bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)


tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    trust_remote_code=True,
)


if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

try:

    lm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        dtype=torch.bfloat16,
        trust_remote_code=True,
    )

except TypeError:

    lm = AutoModelForCausalLM.from_pretrained(
        MODEL_ID,
        quantization_config=bnb_config,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )


lm.eval()

print("Qwen loaded.")

print("Loading answer-clustering embedder...")


embedder = SentenceTransformer(
    EMBED_ID,
    device="cpu",
)


print("Embedder loaded.")


class StopOnStrings(StoppingCriteria):

    def __init__(
        self,
        tokenizer,
        start_length,
        stop_strings,
    ):

        super().__init__()

        self.tokenizer = tokenizer

        self.start_length = start_length

        self.stop_strings = stop_strings


    def __call__(
        self,
        input_ids,
        scores,
        **kwargs,
    ):


        for row in input_ids:

            generated_ids = row[
                self.start_length:
            ]

            generated_text = (
                self.tokenizer.decode(
                    generated_ids,
                    skip_special_tokens=True,
                )
            )

            if not any(
                stop_string in generated_text
                for stop_string in self.stop_strings
            ):

                return False

        return True


SYSTEM_PROMPT = """
You answer multi-hop questions by reasoning ONE factual unit at a time.

Use the supplied evidence to reason toward the answer.

On each turn, output EXACTLY ONE of the following:

<step>
one declarative factual statement
</step>

or

<final>
short final answer
</final>

Rules for <step>:
- Write exactly one factual claim.
- It must be declarative, not a question.
- Keep it short and self-contained.
- Build on the reasoning already produced.
- Do not repeat earlier steps.
- Do not include the final answer inside a reasoning step.
- Output <final> when the reasoning is sufficient to answer the question.

Do not output explanations outside the tags.
""".strip()


def apply_chat_template(
    user_content,
):

    messages = [
        {
            "role": "system",
            "content": SYSTEM_PROMPT,
        },
        {
            "role": "user",
            "content": user_content.strip(),
        },
    ]


    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )


def format_history(
    history_steps,
):

    if not history_steps:
        return "(none yet)"

    return "\n".join(
        f"<step>\n{step.strip()}\n</step>"
        for step in history_steps
    )


def format_evidence(
    evidence,
):

    if not evidence:
        return "(no retrieved evidence)"

    return "\n".join(
        f"[{i + 1}] {sentence}"
        for i, sentence in enumerate(evidence)
    )


def build_step_prompt(
    question,
    evidence,
    history_steps,
):

    user_content = f"""
Evidence:
{format_evidence(evidence)}

Question:
{question}

Reasoning so far:
{format_history(history_steps)}

Produce exactly ONE next reasoning unit now.

Use either:

<step>
one factual statement
</step>

or:

<final>
short final answer
</final>
"""

    return apply_chat_template(
        user_content
    )


def is_yes_no_question(
    question,
):

    q = question.strip().lower()

    return q.startswith(
        (
            "is ",
            "are ",
            "was ",
            "were ",
            "do ",
            "does ",
            "did ",
            "can ",
            "could ",
            "would ",
            "will ",
            "has ",
            "have ",
            "had ",
        )
    )


def build_final_prompt(
    question,
    evidence,
    history_steps,
):

    if is_yes_no_question(question):

        final_rule = (
            "The final answer must be exactly Yes or No."
        )

    else:

        final_rule = (
            "Give only the specific entity, person, place, title, "
            "date, number, nationality, or phrase requested."
        )

    user_content = f"""
Evidence:
{format_evidence(evidence)}

Question:
{question}

Reasoning:
{format_history(history_steps)}

Now give the final answer.

Use exactly:

<final>
short final answer
</final>

Rules:
- {final_rule}
- Do not explain.
- Do not leave the answer blank.
"""

    return apply_chat_template(
        user_content
    )


def clean_text(
    text,
):

    text = str(text).strip()

    text = re.sub(
        r"</?(step|final)>",
        "",
        text,
        flags=re.IGNORECASE,
    )

    text = re.sub(
        r"\s+",
        " ",
        text,
    )

    return text.strip()


def parse_unit(
    text,
    fallback_type="step",
):
    """
    Extract the first reasoning step or final answer.

    During reasoning:
        untagged output -> step

    During final-answer generation:
        untagged output -> final

    No reliability filtering is performed here.
    """

    raw = str(text)


    m = re.search(
        r"<final>\s*(.*?)\s*</final>",
        raw,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if m:

        return {
            "type": "final",
            "text": clean_text(
                m.group(1)
            ),
            "raw": raw,
        }


    m = re.search(
        r"<step>\s*(.*?)\s*</step>",
        raw,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if m:

        return {
            "type": "step",
            "text": clean_text(
                m.group(1)
            ),
            "raw": raw,
        }



    m = re.search(
        r"<final>\s*(.*)",
        raw,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if m:

        content = re.split(
            r"</final>|<step>|</step>",
            m.group(1),
            flags=re.IGNORECASE,
        )[0]

        return {
            "type": "final",
            "text": clean_text(
                content
            ),
            "raw": raw,
        }


    m = re.search(
        r"<step>\s*(.*)",
        raw,
        flags=re.IGNORECASE | re.DOTALL,
    )

    if m:

        content = re.split(
            r"</step>|<final>|</final>",
            m.group(1),
            flags=re.IGNORECASE,
        )[0]

        return {
            "type": "step",
            "text": clean_text(
                content
            ),
            "raw": raw,
        }


    cleaned = clean_text(
        raw
    )


    if not cleaned:

        return {
            "type": "none",
            "text": "",
            "raw": raw,
        }


    m = re.search(
        r"(?:therefore,\s*)?"
        r"(?:the\s+)?answer"
        r"(?:\s+to\s+the\s+question)?"
        r"\s+is\s*[:\-]?\s*(.+)",
        cleaned,
        flags=re.IGNORECASE,
    )

    if m:

        return {
            "type": "final",
            "text": clean_text(
                m.group(1)
            ),
            "raw": raw,
        }


    return {
        "type": fallback_type,
        "text": cleaned,
        "raw": raw,
    }



def flatten_context(
    record,
):

    context = record.get(
        "context",
        {},
    )


    if not isinstance(
        context,
        dict,
    ):

        raise ValueError(
            "Unexpected context format for "
            f"question_index={record.get('question_index')}"
        )


    groups = context.get(
        "sentences",
        [],
    )



    if (
        groups
        and isinstance(
            groups[0],
            str,
        )
    ):

        groups = [
            groups
        ]


    sentences = []


    for group in groups:

        for sentence in group:

            sentence = (
                str(sentence)
                .strip()
            )

            if sentence:

                sentences.append(
                    sentence
                )


    return sentences


def bm25_question_evidence(
    record,
    question,
    k=RETRIEVE_K,
):

    sentences = flatten_context(
        record
    )


    if not sentences:
        return []


    def bm25_tokens(
        text,
    ):

        return re.findall(
            r"\w+",
            str(text).lower(),
        )


    bm25 = BM25Okapi(
        [
            bm25_tokens(sentence)
            for sentence in sentences
        ]
    )


    scores = bm25.get_scores(
        bm25_tokens(question)
    )


    order = np.argsort(
        scores
    )[::-1][:k]


    return [
        sentences[int(i)]
        for i in order
    ]



@torch.no_grad()
def generate_one_reasoning_unit(
    question,
    evidence,
    history_steps,
):

    prompt = build_step_prompt(
        question,
        evidence,
        history_steps,
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(
        lm.device
    )


    start_length = (
        inputs[
            "input_ids"
        ].shape[-1]
    )


    stop = StoppingCriteriaList(
        [
            StopOnStrings(
                tokenizer,
                start_length,
                [
                    "</step>",
                    "</final>",
                ],
            )
        ]
    )


    output = lm.generate(
        **inputs,



        do_sample=False,

        num_return_sequences=1,


        max_new_tokens=STEP_MAX_NEW_TOKENS,

        stopping_criteria=stop,

        pad_token_id=tokenizer.eos_token_id,

        eos_token_id=tokenizer.eos_token_id,
    )


    generated = output[
        0,
        start_length:
    ]


    text = tokenizer.decode(
        generated,
        skip_special_tokens=True,
    )


    return parse_unit(
        text,
        fallback_type="step",
    )



@torch.no_grad()
def sample_final_batch(
    question,
    evidence,
    history_steps,
    n,
):

    prompt = build_final_prompt(
        question,
        evidence,
        history_steps,
    )


    inputs = tokenizer(
        prompt,
        return_tensors="pt",
    ).to(
        lm.device
    )


    start_length = (
        inputs[
            "input_ids"
        ].shape[-1]
    )


    stop = StoppingCriteriaList(
        [
            StopOnStrings(
                tokenizer,
                start_length,
                [
                    "</final>"
                ],
            )
        ]
    )


    outputs = lm.generate(
        **inputs,

        do_sample=True,

        temperature=FINAL_TEMPERATURE,

        top_p=FINAL_TOP_P,

        max_new_tokens=FINAL_MAX_NEW_TOKENS,

        num_return_sequences=n,

        stopping_criteria=stop,

        pad_token_id=tokenizer.eos_token_id,

        eos_token_id=tokenizer.eos_token_id,
    )


    generated = outputs[
        :,
        start_length:
    ]


    decoded = tokenizer.batch_decode(
        generated,
        skip_special_tokens=True,
    )


    return [
        parse_unit(
            text,
            fallback_type="final",
        )
        for text in decoded
    ]


def generate_final_samples(
    question,
    evidence,
    history_steps,
):

    answers = []


    for _ in range(
        EMPTY_FINAL_RETRY_ATTEMPTS
        +
        1
    ):

        remaining = (
            FINAL_SAMPLES
            -
            len(answers)
        )


        if remaining <= 0:
            break


        units = sample_final_batch(
            question,
            evidence,
            history_steps,
            remaining,
        )


        for unit in units:

            answer = (
                unit.get(
                    "text",
                    "",
                )
                .strip()
            )


            if answer:

                answers.append(
                    answer
                )


    answers = answers[
        :FINAL_SAMPLES
    ]


    while (
        len(answers)
        <
        FINAL_SAMPLES
    ):

        answers.append(
            ""
        )


    return answers



def normalize_answer(
    text,
):

    text = (
        str(text)
        .lower()
        .strip()
    )




    text = re.sub(
        r"\b(a|an|the)\b",
        " ",
        text,
    )




    text = re.sub(
        r"[^\w\s]",
        " ",
        text,
    )


    text = re.sub(
        r"\s+",
        " ",
        text,
    )


    return text.strip()




def _shannon_entropy(
    counts,
):

    counts = [
        count
        for count in counts
        if count > 0
    ]


    if len(counts) <= 1:

        return (
            0.0,
            0.0,
        )


    total = sum(
        counts
    )


    probs = (
        np.asarray(
            counts,
            dtype=float,
        )
        /
        total
    )


    entropy = float(
        -np.sum(
            probs
            *
            np.log2(
                probs
                +
                1e-12
            )
        )
    )


    normalized_entropy = float(
        entropy
        /
        np.log2(
            len(counts)
        )
    )


    return (
        entropy,
        normalized_entropy,
    )



def cluster_answers(
    answers,
):

    clean = [
        str(answer).strip()
        for answer in answers
        if str(answer).strip()
    ]


    if not clean:
        return []


    norms = [
        normalize_answer(
            answer
        )
        for answer in clean
    ]


    embeddings = embedder.encode(
        clean,
        normalize_embeddings=True,
        convert_to_numpy=True,
        show_progress_bar=False,
    )


    clusters = []


    for i, norm_i in enumerate(
        norms
    ):

        placed = False


        for cluster in clusters:

            representative_norm = (
                cluster["norm"]
            )



            same = (
                norm_i
                ==
                representative_norm

                or

                (
                    norm_i
                    and
                    representative_norm
                    and
                    (
                        norm_i
                        in
                        representative_norm

                        or

                        representative_norm
                        in
                        norm_i
                    )
                )
            )




            if not same:

                similarity = float(
                    np.dot(
                        embeddings[i],
                        embeddings[
                            cluster[
                                "idx0"
                            ]
                        ],
                    )
                )


                same = (
                    similarity
                    >=
                    ANSWER_MERGE_SIM
                )


            if same:

                cluster[
                    "idxs"
                ].append(
                    i
                )

                placed = True

                break


        if not placed:

            clusters.append(
                {
                    "norm":
                        norm_i,

                    "idx0":
                        i,

                    "idxs":
                        [i],
                }
            )


    return clusters



def self_consistency_stats(
    answers,
):

    answers = [
        str(answer).strip()
        for answer in answers
        if str(answer).strip()
    ]


    if not answers:

        return {
            "majority_answer":
                "",

            "self_consistency":
                0.0,

            "norm_entropy":
                1.0,

            "num_clusters":
                0,

            "n_valid_samples":
                0,
        }


    clusters = cluster_answers(
        answers
    )


    sizes = [
        len(
            cluster[
                "idxs"
            ]
        )
        for cluster in clusters
    ]


    largest_cluster = max(
        clusters,
        key=lambda cluster:
            len(
                cluster[
                    "idxs"
                ]
            ),
    )


    majority_answer = answers[
        largest_cluster[
            "idxs"
        ][0]
    ]


    _, normalized_entropy = (
        _shannon_entropy(
            sizes
        )
    )


    return {
        "majority_answer":
            majority_answer,

        "self_consistency":
            max(sizes)
            /
            len(answers),

        "norm_entropy":
            normalized_entropy,

        "num_clusters":
            len(clusters),

        "n_valid_samples":
            len(answers),
    }



@torch.no_grad()
def answer_logprob(
    question,
    evidence,
    history_steps,
    answer,
):
    """
    Mean token log-probability of the selected final answer
    conditioned on:

        question
        + retrieved evidence
        + fixed reasoning history

    The answer tokens themselves are excluded from the conditioning
    context and evaluated autoregressively.
    """

    answer = (
        str(answer)
        .strip()
    )


    if not answer:

        return float(
            "nan"
        )


    base_prompt = build_final_prompt(
        question,
        evidence,
        history_steps,
    )




    prefix = (
        base_prompt
        +
        "<final>\n"
    )



    prefix_ids = tokenizer(
        prefix,
        return_tensors="pt",
        add_special_tokens=False,
    )[
        "input_ids"
    ].to(
        lm.device
    )


    answer_ids = tokenizer(
        answer,
        return_tensors="pt",
        add_special_tokens=False,
    )[
        "input_ids"
    ].to(
        lm.device
    )


    if (
        answer_ids.shape[1]
        ==
        0
    ):

        return float(
            "nan"
        )


    input_ids = torch.cat(
        [
            prefix_ids,
            answer_ids,
        ],
        dim=1,
    )


    attention_mask = (
        torch.ones_like(
            input_ids
        )
    )


    logits = lm(
        input_ids=input_ids,
        attention_mask=attention_mask,
    ).logits.float()


    prefix_length = (
        prefix_ids.shape[1]
    )


    answer_length = (
        answer_ids.shape[1]
    )




    answer_logits = logits[
        :,
        prefix_length - 1:
        prefix_length + answer_length - 1,
        :
    ]


    log_probs = torch.log_softmax(
        answer_logits,
        dim=-1,
    )


    token_log_probs = (
        log_probs.gather(
            -1,
            answer_ids.unsqueeze(-1),
        )
        .squeeze(-1)
    )


    if (
        token_log_probs.numel()
        ==
        0
    ):

        return float(
            "nan"
        )


    return float(
        token_log_probs
        .mean()
        .item()
    )



def load_hotpot_pool():


    old_pool_path = (
        f"{DATA_DIR}/"
        "hedge_pre_rl_1000_full.json"
    )


    test_path = (
        f"{DATA_DIR}/"
        "dpo_test_questions.json"
    )


    if not os.path.exists(
        old_pool_path
    ):

        raise FileNotFoundError(
            old_pool_path
        )


    if not os.path.exists(
        test_path
    ):

        raise FileNotFoundError(
            test_path
        )


    with open(
        old_pool_path,
        "r",
        encoding="utf-8",
    ) as f:

        old_pool = json.load(
            f
        )


    with open(
        test_path,
        "r",
        encoding="utf-8",
    ) as f:

        test_manifest = json.load(
            f
        )


    test_ids = {
        int(
            item[
                "question_index"
            ]
        )
        for item
        in test_manifest
    }


    print(
        "Loading HotpotQA validation split..."
    )


    hotpot_dataset = load_dataset(
        "hotpotqa/hotpot_qa",
        "distractor",
        split="validation",
    )



    question_to_example = {
        example["question"]:
            example
        for example
        in hotpot_dataset
    }


    records = []


    for old_record in old_pool:

        question_index = int(
            old_record[
                "question_index"
            ]
        )


        question = (
            old_record[
                "question"
            ]
        )


        example = (
            question_to_example.get(
                question
            )
        )


        if example is None:

            raise KeyError(
                "Cannot recover HotpotQA context "
                f"for question_index={question_index}\n"
                f"{question}"
            )


        records.append(
            {
                "question_index":
                    question_index,

                "question":
                    question,

                "gold_answer":
                    example[
                        "answer"
                    ],

                "context":
                    example[
                        "context"
                    ],

                "split":
                    (
                        "test"
                        if question_index
                        in test_ids
                        else
                        "dev"
                    ),
            }
        )



    assert len(records) == 1000

    assert len(test_ids) == 200


    assert (
        sum(
            record["split"]
            ==
            "test"

            for record
            in records
        )
        ==
        200
    )


    assert (
        sum(
            record["split"]
            ==
            "dev"

            for record
            in records
        )
        ==
        800
    )


    return records




def get_gold_2wiki(
    record,
):

    for key in [
        "gold_answer",
        "answer",
        "gold",
    ]:

        value = record.get(
            key
        )


        if (
            value is not None
            and
            str(value).strip()
        ):

            return (
                str(value)
                .strip()
            )


    raise KeyError(
        "No gold answer found for "
        f"question_index={record.get('question_index')}"
    )


def load_2wiki_pool():

    full_path = (
        f"{DATA_DIR}/"
        "2wiki_full.json"
    )


    test_path = (
        f"{DATA_DIR}/"
        "2wiki_test_questions.json"
    )


    if not os.path.exists(
        full_path
    ):

        raise FileNotFoundError(
            full_path
        )


    if not os.path.exists(
        test_path
    ):

        raise FileNotFoundError(
            test_path
        )


    with open(
        full_path,
        "r",
        encoding="utf-8",
    ) as f:

        raw_records = (
            json.load(
                f
            )
        )


    with open(
        test_path,
        "r",
        encoding="utf-8",
    ) as f:

        test_manifest = (
            json.load(
                f
            )
        )


    test_ids = {
        int(
            item[
                "question_index"
            ]
        )
        for item
        in test_manifest
    }


    records = []


    for raw_record in raw_records:

        question_index = int(
            raw_record[
                "question_index"
            ]
        )


        if not isinstance(
            raw_record.get(
                "context"
            ),
            dict,
        ):

            raise ValueError(
                "Unexpected 2Wiki context "
                f"for question_index={question_index}"
            )


        records.append(
            {
                "question_index":
                    question_index,

                "question":
                    raw_record[
                        "question"
                    ],

                "gold_answer":
                    get_gold_2wiki(
                        raw_record
                    ),

                "context":
                    raw_record[
                        "context"
                    ],

                "split":
                    (
                        "test"
                        if question_index
                        in test_ids
                        else
                        "dev"
                    ),
            }
        )




    assert len(records) == 1000

    assert len(test_ids) == 200


    assert (
        sum(
            record["split"]
            ==
            "test"

            for record
            in records
        )
        ==
        200
    )


    assert (
        sum(
            record["split"]
            ==
            "dev"

            for record
            in records
        )
        ==
        800
    )


    return records



def seed_for_question(
    dataset_name,
    question_index,
):

    # Dataset-specific offset prevents identical sampling streams
    # across HotpotQA and 2Wiki when question_index is identical.

    dataset_offset = (
        0
        if dataset_name
        ==
        "hotpot"
        else
        100_000
    )


    seed = (
        SEED
        +
        dataset_offset
        +
        int(
            question_index
        )
    )


    random.seed(
        seed
    )

    np.random.seed(
        seed
    )

    torch.manual_seed(
        seed
    )


    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(
            seed
        )



def is_conclusion_like_step(
    text,
):
    """
    Detect cases where the model accidentally places an explicit
    final conclusion inside <step>.

    These are separated from the intermediate reasoning trajectory.

    This is formatting/post-processing only.
    It is NOT verifier-based reliability filtering.
    """

    text = (
        str(text)
        .strip()
    )



    if re.match(
        r"^(?:the\s+)?answer"
        r"(?:\s+to\s+the\s+question)?"
        r"\s+is\b",
        text,
        flags=re.IGNORECASE,
    ):

        return True


    if re.match(
        r"^(therefore|thus|hence|so)"
        r"(?:,|\s)+",
        text,
        flags=re.IGNORECASE,
    ):

        return True


    return False


def clean_loop_final_answer(
    text,
):

    text = (
        str(text)
        .strip()
    )




    text = re.sub(
        r"^(therefore|thus|hence|so)"
        r"(?:,|\s)+",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()




    text = re.sub(
        r"^(?:the\s+)?answer"
        r"(?:\s+to\s+the\s+question)?"
        r"\s+is\s*[:\-]?\s*",
        "",
        text,
        flags=re.IGNORECASE,
    ).strip()


    text = (
        text
        .strip(
            " \"'`*<>"
        )
        .rstrip(
            " ."
        )
    )


    return text



def run_question(
    dataset_name,
    record,
):

    question_index = int(
        record[
            "question_index"
        ]
    )


    question = (
        record[
            "question"
        ]
    )


    seed_for_question(
        dataset_name,
        question_index,
    )




    evidence = bm25_question_evidence(
        record,
        question,
        RETRIEVE_K,
    )




    history = []

    generation_trace = []

    loop_final_answer = ""

    stop_reason = "max_steps"


    for step_index in range(
        1,
        MAX_STEPS + 1,
    ):

        unit = (
            generate_one_reasoning_unit(
                question,
                evidence,
                history,
            )
        )


        unit_type = (
            unit.get(
                "type",
                "none",
            )
        )


        text = (
            unit.get(
                "text",
                "",
            )
            .strip()
        )


        generation_trace.append(
            {
                "step_index":
                    step_index,

                "type":
                    unit_type,

                "text":
                    text,
            }
        )




        if (
            unit_type
            ==
            "none"

            or

            not text
        ):

            stop_reason = (
                "empty_generation"
            )

            break




        if (
            unit_type
            ==
            "final"
        ):

            loop_final_answer = (
                clean_loop_final_answer(
                    text
                )
            )

            stop_reason = (
                "stop_final"
            )

            break




        if is_conclusion_like_step(
            text
        ):

            loop_final_answer = (
                clean_loop_final_answer(
                    text
                )
            )

            stop_reason = (
                "conclusion_as_step"
            )

            break


        history.append(
            text
        )



    final_answers = (
        generate_final_samples(
            question,
            evidence,
            history,
        )
    )




    confidence_stats = (
        self_consistency_stats(
            final_answers
        )
    )


    majority_answer = (
        confidence_stats[
            "majority_answer"
        ]
    )



    mean_answer_logprob = (
        answer_logprob(
            question,
            evidence,
            history,
            majority_answer,
        )
    )



    return {

        "question_index":
            question_index,

        "split":
            record[
                "split"
            ],

        "question":
            question,


        "gold_answer":
            record[
                "gold_answer"
            ],



        "question_evidence":
            evidence,



        "selected_steps":
            history,

        "num_steps":
            len(
                history
            ),

        "stop_reason":
            stop_reason,

        "loop_final_answer":
            loop_final_answer,

        "generation_trace":
            generation_trace,



        "final_answers":
            final_answers,

        "majority_answer":
            majority_answer,

        "self_consistency":
            float(
                confidence_stats[
                    "self_consistency"
                ]
            ),

        "answer_norm_entropy":
            float(
                confidence_stats[
                    "norm_entropy"
                ]
            ),

        "answer_logprob":
            mean_answer_logprob,

        "num_answer_clusters":
            int(
                confidence_stats[
                    "num_clusters"
                ]
            ),

        "num_valid_final_samples":
            int(
                confidence_stats[
                    "n_valid_samples"
                ]
            ),




        "generation_protocol": {

            "version":
                "passive_iterative_v2",

            "reasoning":
                "iterative_greedy_passive",

            "step_sampling":
                False,

            "verifier_during_reasoning":
                False,

            "candidate_selection":
                False,

            "gold_evidence":
                False,

            "question_retrieve_k":
                RETRIEVE_K,

            "max_steps":
                MAX_STEPS,

            "final_samples":
                FINAL_SAMPLES,

            "final_temperature":
                FINAL_TEMPERATURE,

            "final_top_p":
                FINAL_TOP_P,
        },
    }


def run_dataset(
    dataset_name,
):

    print()

    print(
        "=" * 80
    )

    print(
        "RUNNING CLEAN PASSIVE ITERATIVE GENERATION: "
        f"{dataset_name.upper()}"
    )

    print(
        "=" * 80
    )



    if (
        dataset_name
        ==
        "hotpot"
    ):

        records = (
            load_hotpot_pool()
        )


        progress_path = (
            f"{DATA_DIR}/"
            "hotpot_passive_iterative_v2_progress.json"
        )


        final_path = (
            f"{DATA_DIR}/"
            "hotpot_passive_iterative_v2_1000.json"
        )


    elif (
        dataset_name
        ==
        "2wiki"
    ):

        records = (
            load_2wiki_pool()
        )


        progress_path = (
            f"{DATA_DIR}/"
            "2wiki_passive_iterative_v2_progress.json"
        )


        final_path = (
            f"{DATA_DIR}/"
            "2wiki_passive_iterative_v2_1000.json"
        )


    else:

        raise ValueError(
            f"Unknown dataset: {dataset_name}"
        )




    if os.path.exists(
        progress_path
    ):

        with open(
            progress_path,
            "r",
            encoding="utf-8",
        ) as f:

            progress = (
                json.load(
                    f
                )
            )

    else:

        progress = {}


    print(
        f"Total records: "
        f"{len(records)}"
    )

    print(
        f"Already completed: "
        f"{len(progress)}"
    )

    print(
        "Remaining: "
        f"{len(records) - len(progress)}"
    )




    for record in tqdm(
        records
    ):

        question_index = int(
            record[
                "question_index"
            ]
        )


        key = str(
            question_index
        )


        if key in progress:

            continue


        try:

            result = (
                run_question(
                    dataset_name,
                    record,
                )
            )


        except Exception:

            # Save everything completed before propagating error.

            with open(
                progress_path,
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    progress,
                    f,
                    indent=2,
                    ensure_ascii=False,
                )


            print()

            print(
                "FAILED ON:"
            )

            print(
                "dataset =",
                dataset_name,
            )

            print(
                "question_index =",
                question_index,
            )

            print(
                "question =",
                record[
                    "question"
                ],
            )


            raise


        progress[
            key
        ] = result




        if (
            len(progress)
            %
            SAVE_EVERY
            ==
            0
        ):

            with open(
                progress_path,
                "w",
                encoding="utf-8",
            ) as f:

                json.dump(
                    progress,
                    f,
                    indent=2,
                    ensure_ascii=False,
                )


            print(
                f"{dataset_name}: "
                f"{len(progress)}/1000 saved"
            )




    with open(
        progress_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            progress,
            f,
            indent=2,
            ensure_ascii=False,
        )




    final_records = [
        progress[
            str(
                int(
                    record[
                        "question_index"
                    ]
                )
            )
        ]
        for record
        in records
    ]




    with open(
        final_path,
        "w",
        encoding="utf-8",
    ) as f:

        json.dump(
            final_records,
            f,
            indent=2,
            ensure_ascii=False,
        )



    assert (
        len(
            final_records
        )
        ==
        1000
    )


    assert (
        sum(
            record[
                "split"
            ]
            ==
            "dev"

            for record
            in final_records
        )
        ==
        800
    )


    assert (
        sum(
            record[
                "split"
            ]
            ==
            "test"

            for record
            in final_records
        )
        ==
        200
    )


    # Confirm every record belongs to this exact generation version.

    assert all(
        record[
            "generation_protocol"
        ][
            "version"
        ]
        ==
        "passive_iterative_v2"

        for record
        in final_records
    )



    print()

    print(
        f"FINISHED {dataset_name.upper()}"
    )

    print(
        "Saved:",
        final_path,
    )


    no_final = sum(
        not str(
            record[
                "majority_answer"
            ]
        ).strip()

        for record
        in final_records
    )


    mean_steps = float(
        np.mean(
            [
                record[
                    "num_steps"
                ]
                for record
                in final_records
            ]
        )
    )


    stop_counts = Counter(
        record[
            "stop_reason"
        ]
        for record
        in final_records
    )


    sc_values = np.asarray(
        [
            record[
                "self_consistency"
            ]
            for record
            in final_records
        ],
        dtype=float,
    )


    entropy_values = np.asarray(
        [
            record[
                "answer_norm_entropy"
            ]
            for record
            in final_records
        ],
        dtype=float,
    )


    print(
        "No majority final answer:",
        no_final,
    )

    print(
        "Mean reasoning steps:",
        round(
            mean_steps,
            3,
        ),
    )

    print(
        "Stop reasons:",
        dict(
            stop_counts
        ),
    )

    print(
        "Mean self-consistency:",
        round(
            float(
                sc_values.mean()
            ),
            4,
        ),
    )

    print(
        "Fraction self-consistency = 1:",
        round(
            float(
                np.mean(
                    sc_values
                    ==
                    1.0
                )
            ),
            4,
        ),
    )

    print(
        "Mean normalized entropy:",
        round(
            float(
                entropy_values.mean()
            ),
            4,
        ),
    )


    return final_records




all_outputs = {}


for dataset_name in RUN_DATASETS:

    all_outputs[
        dataset_name
    ] = run_dataset(
        dataset_name
    )




print()

print(
    "=" * 80
)

print(
    "BOTH DATASETS COMPLETE"
)

print(
    "=" * 80
)


print(
    "HotpotQA:"
)

print(
    f"{DATA_DIR}/"
    "hotpot_passive_iterative_v2_1000.json"
)


print(
    "\n2WikiMultiHopQA:"
)

print(
    f"{DATA_DIR}/"
    "2wiki_passive_iterative_v2_1000.json"
)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loading Qwen...


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

...

Total records: 1000
Already completed: 1000
Remaining: 0


  0%|          | 0/1000 [00:00<?, ?it/s]


FINISHED HOTPOT
Saved: /content/drive/MyDrive/hedge_run/hotpot_passive_iterative_v2_1000.json
No majority final answer: 0
Mean reasoning steps: 2.057
Stop reasons: {'stop_final': 914, 'max_steps': 81, 'conclusion_as_step': 5}
Mean self-consistency: 0.9654
Fraction self-consistency = 1: 0.887
Mean normalized entropy: 0.0844

RUNNING CLEAN PASSIVE ITERATIVE GENERATION: 2WIKI
Total records: 1000
Already completed: 195
Remaining: 805


  0%|          | 0/1000 [00:00<?, ?it/s]

2wiki: 200/1000 saved
2wiki: 205/1000 saved
2wiki: 210/1000 saved
2wiki: 215/1000 saved
2wiki: 220/1000 saved
2wiki: 225/1000 saved
2wiki: 230/1000 saved
2wiki: 235/1000 saved
2wiki: 240/1000 saved
2wiki: 245/1000 saved
2wiki: 250/1000 saved
2wiki: 255/1000 saved
2wiki: 260/1000 saved
2wiki: 265/1000 saved
2wiki: 270/1000 saved
2wiki: 275/1000 saved
2wiki: 280/1000 saved
2wiki: 285/1000 saved
2wiki: 290/1000 saved
2wiki: 295/1000 saved
2wiki: 300/1000 saved
2wiki: 305/1000 saved
2wiki: 310/1000 saved
2wiki: 315/1000 saved
2wiki: 320/1000 saved
2wiki: 325/1000 saved
2wiki: 330/1000 saved
2wiki: 335/1000 saved
2wiki: 340/1000 saved
2wiki: 345/1000 saved
2wiki: 350/1000 saved
2wiki: 355/1000 saved
2wiki: 360/1000 saved
2wiki: 365/1000 saved
2wiki: 370/1000 saved
2wiki: 375/1000 saved
2wiki: 380/1000 saved
2wiki: 385/1000 saved
2wiki: 390/1000 saved
2wiki: 395/1000 saved
2wiki: 400/1000 saved
2wiki: 405/1000 saved
2wiki: 410/1000 saved
2wiki: 415/1000 saved
2wiki: 420/1000 saved
2wiki: 425

In [ ]:
import json
import numpy as np
from collections import Counter

PATH = "/content/drive/MyDrive/hedge_run/hotpot_passive_iterative_v2_1000.json"

with open(PATH, "r", encoding="utf-8") as f:
    rows = json.load(f)

print("=" * 70)
print("HOTPOTQA GENERATION SANITY CHECK")
print("=" * 70)

print("N:", len(rows))


steps = np.array(
    [r["num_steps"] for r in rows],
    dtype=int
)

print("\nREASONING LENGTH")
print("mean:", steps.mean())
print("distribution:", Counter(steps))


print("\nSTOP REASONS")
print(Counter(
    r["stop_reason"]
    for r in rows
))


missing = [
    r for r in rows
    if not str(r["majority_answer"]).strip()
]

print("\nFINAL ANSWERS")
print("missing majority answers:", len(missing))



sc = np.array(
    [r["self_consistency"] for r in rows],
    dtype=float
)

print("\nSELF-CONSISTENCY")
print("mean:", sc.mean())
print("min:", sc.min())
print("fraction = 1.0:", np.mean(sc == 1.0))
print("fraction < 0.8:", np.mean(sc < 0.8))
print("unique values:", sorted(set(sc)))


ent = np.array(
    [r["answer_norm_entropy"] for r in rows],
    dtype=float
)

print("\nNORMALISED ENTROPY")
print("mean:", ent.mean())
print("max:", ent.max())
print("fraction = 0:", np.mean(ent == 0.0))
print("fraction > 0.5:", np.mean(ent > 0.5))



lp = np.array(
    [
        r["answer_logprob"]
        for r in rows
        if r["answer_logprob"] is not None
        and np.isfinite(r["answer_logprob"])
    ],
    dtype=float
)

print("\nANSWER LOGPROB")
print("valid:", len(lp))
print("mean:", lp.mean())
print("min:", lp.min())
print("max:", lp.max())


def norm(x):
    x = str(x).lower().strip()
    x = "".join(c for c in x if c.isalnum() or c.isspace())
    return " ".join(x.split())

both = [
    r for r in rows
    if str(r["loop_final_answer"]).strip()
    and str(r["majority_answer"]).strip()
]

different = [
    r for r in both
    if norm(r["loop_final_answer"])
    != norm(r["majority_answer"])
]

print("\nLOOP FINAL vs MAJORITY ANSWER")
print("both available:", len(both))
print("different:", len(different))
print(
    "fraction different:",
    len(different) / len(both) if both else 0
)

HOTPOTQA GENERATION SANITY CHECK
N: 1000

REASONING LENGTH
mean: 2.057
distribution: Counter({np.int64(2): 521, np.int64(1): 235, np.int64(3): 152, np.int64(4): 81, np.int64(0): 11})

STOP REASONS
Counter({'stop_final': 914, 'max_steps': 81, 'conclusion_as_step': 5})

FINAL ANSWERS
missing majority answers: 0

SELF-CONSISTENCY
mean: 0.9654
min: 0.2
fraction = 1.0: 0.887
fraction < 0.8: 0.059
unique values: [np.float64(0.2), np.float64(0.3), np.float64(0.4), np.float64(0.5), np.float64(0.6), np.float64(0.7), np.float64(0.8), np.float64(0.9), np.float64(1.0)]

NORMALISED ENTROPY
mean: 0.08437200169855666
max: 0.9999999999971146
fraction = 0: 0.887
fraction > 0.5: 0.089

ANSWER LOGPROB
valid: 1000
mean: -0.11567285957712212
min: -6.125012397766113
max: 0.0

LOOP FINAL vs MAJORITY ANSWER
both available: 919
different: 482
fraction different: 0.5244831338411317


In [ ]:
import json
import re
import random

PATH = (
    "/content/drive/MyDrive/hedge_run/"
    "hotpot_passive_iterative_v2_1000.json"
)

with open(PATH, "r", encoding="utf-8") as f:
    rows = json.load(f)


def norm(x):
    x = str(x).lower().strip()
    x = re.sub(r"[^\w\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


both = [
    r for r in rows
    if str(r["loop_final_answer"]).strip()
    and str(r["majority_answer"]).strip()
]


exact_same = 0
containment_same = 0
real_candidates = []

for r in both:

    loop = norm(r["loop_final_answer"])
    maj = norm(r["majority_answer"])

    if loop == maj:
        exact_same += 1
        containment_same += 1

    elif maj in loop or loop in maj:
        containment_same += 1

    else:
        real_candidates.append(r)


print("Both available:", len(both))

print(
    "Exact match:",
    exact_same,
    f"({exact_same / len(both):.3f})"
)

print(
    "Exact OR containment:",
    containment_same,
    f"({containment_same / len(both):.3f})"
)

print(
    "Neither exact nor containment:",
    len(real_candidates),
    f"({len(real_candidates) / len(both):.3f})"
)


print("\nExamples of genuine-looking differences:")

random.seed(42)

for r in random.sample(
    real_candidates,
    min(10, len(real_candidates))
):
    print("\nQUESTION:")
    print(r["question"])

    print("Loop final:")
    print(r["loop_final_answer"])

    print("Majority:")
    print(r["majority_answer"])

    print("Gold:")
    print(r["gold_answer"])

Both available: 919
Exact match: 437 (0.476)
Exact OR containment: 755 (0.822)
Neither exact nor containment: 164 (0.178)

Examples of genuine-looking differences:

QUESTION:
A man who played in the 1986 FIFA world cup played for what team during the 1982 Scottish League Cup Final?
Loop final:
Dunfermline
Majority:
Kilmarnock
Gold:
Celtic

QUESTION:
Are Eve Beglarian and Zach Bogosian both of Armenian descent?
Loop final:
Yes, both Eve Beglarian and Zach Bogosian are of Armenian descent
Majority:
No
Gold:
yes

QUESTION:
In what year was the most famous statute at Po Lin Monastery built?
Loop final:
The evidence does not provide a specific year for the construction of Tian Tan Buddha
Majority:
1956
Gold:
Tian Tan Buddha

QUESTION:
Prominent Danish Tibetologist Per Kjeld Sørensen is a professor of Central Asian Studies at Leipzig University that was founded by who?
Loop final:
Leipzig University was founded by the city of Leipzig
Majority:
none
Gold:
Frederick I, Elector of Saxony

QUEST

In [ ]:
from google.colab import runtime
runtime.unassign()